In [71]:
import pandas as pd

# -------------------------
# Step 0: Load the dataset
# -------------------------
df = pd.read_csv("T_ONTIME_REPORTING.csv")
print("Original Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nFirst 5 rows:\n", df.head())
print("\nData types:\n", df.dtypes)

# -------------------------
# Step 1: Skip cancelled/diverted filter (not present in new dataset)
# -------------------------
print("\nSkipping cancelled/diverted filter: columns not present.")

# -------------------------
# Step 2: Convert FL_DATE safely
# -------------------------
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], errors="coerce")
print("\nFL_DATE after conversion:\n", df["FL_DATE"].head())
print("FL_DATE dtype:", df["FL_DATE"].dtype)
print("Rows with invalid FL_DATE:", df["FL_DATE"].isna().sum())

# -------------------------
# Step 3: Convert time columns to proper strings
# -------------------------
def convert_time_column(col):
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(0).astype(int)
    df[col] = df[col].astype(str).str.zfill(4)

time_columns = ["DEP_TIME", "ARR_TIME", "CRS_DEP_TIME", "CRS_ARR_TIME"]
for col in time_columns:
    convert_time_column(col)

# -------------------------
# Step 4: Combine FL_DATE + times → datetime
# -------------------------
df["scheduled_departure"] = pd.to_datetime(
    df["FL_DATE"].astype(str) + " " + df["CRS_DEP_TIME"],
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

df["actual_departure"] = pd.to_datetime(
    df["FL_DATE"].astype(str) + " " + df["DEP_TIME"],
    format="%Y-%m-%d %H%M",
    errors="coerce"
)

# -------------------------
# Step 5: Create target variables
# -------------------------
df["dep_delayed_15"] = (df["DEP_DELAY"] > 15).astype(int)
df["arr_delay_min"] = df["ARR_DELAY_NEW"]

# -------------------------
# Step 6: Feature engineering
# -------------------------
df["hour_of_day"] = df["scheduled_departure"].dt.hour
df["day_of_week"] = df["scheduled_departure"].dt.dayofweek
df["month"] = df["scheduled_departure"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)
df["route"] = df["ORIGIN"] + "_" + df["DEST"]
peak_hours = [6,7,8,17,18]
df["is_peak_hour"] = df["hour_of_day"].isin(peak_hours).astype(int)

# Operational numeric features
for col in ["TAXI_OUT", "TAXI_IN", "DISTANCE"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Categorical aircraft
if "TAIL_NUM" in df.columns:
    df["TAIL_NUM"] = df["TAIL_NUM"].astype(str)

# -------------------------
# Step 7: Drop rows with missing actual_departure
# -------------------------
df = df.dropna(subset=["actual_departure"])
print("\nShape after dropping missing actual_departure:", df.shape)

# -------------------------
# Step 8: Check missing values
# -------------------------
missing = df.isnull().sum().sort_values(ascending=False)
print("\nTop 15 columns with missing values:\n", missing.head(15))

# -------------------------
# Step 9: Preview final features
# -------------------------
preview_cols = ["scheduled_departure","actual_departure","dep_delayed_15","arr_delay_min",
                "hour_of_day","day_of_week","month","is_weekend","is_peak_hour","route",
                "TAIL_NUM","TAXI_OUT","TAXI_IN","DISTANCE"]
preview_cols = [c for c in preview_cols if c in df.columns]
print("\nPreview features for modelling:\n", df[preview_cols].head())

# -------------------------
# Step 10: Save cleaned dataset
# -------------------------
df.to_csv("cleaned_flights.csv", index=False)
print("\nCleaned dataset saved as 'cleaned_flights.csv'")


Original Shape: (539747, 30)

Columns:
 Index(['YEAR', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME',
       'DEST_AIRPORT_ID', 'DEST', 'DEST_CITY_NAME', 'CRS_DEP_TIME', 'DEP_TIME',
       'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN',
       'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
       'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='object')

First 5 rows:
    YEAR  DAY_OF_WEEK               FL_DATE OP_UNIQUE_CARRIER TAIL_NUM  \
0  2025            1  1/6/2025 12:00:00 AM                AA   N102NN   
1  2025            1  1/6/2025 12:00:00 AM                AA   N102NN   
2  2025            1  1/6/2025 12:00:00 AM                AA   N102UW   
3  2025            1  1/6/2025 12:00:00 AM                AA   N102UW   
4  2025            1  1/6/2025 12:00:00 AM                

C:\Users\noorf\AppData\Local\Temp\ipykernel_20444\1405064706.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], errors="coerce")



Shape after dropping missing actual_departure: (539714, 40)

Top 15 columns with missing values:
 LATE_AIRCRAFT_DELAY    441594
CARRIER_DELAY          441594
WEATHER_DELAY          441594
NAS_DELAY              441594
SECURITY_DELAY         441594
ARR_DELAY_NEW           17478
ARR_DEL15               17478
ARR_DELAY               17478
arr_delay_min           17478
DEP_DELAY               15923
DEP_DELAY_NEW           15923
DEP_DEL15               15923
dep_delayed_15              0
actual_departure            0
scheduled_departure         0
dtype: int64

Preview features for modelling:
   scheduled_departure    actual_departure  dep_delayed_15  arr_delay_min  \
0 2025-01-06 13:21:00 2025-01-06 13:16:00               0            0.0   
1 2025-01-06 08:29:00 2025-01-06 08:24:00               0            0.0   
2 2025-01-06 18:39:00 2025-01-06 18:44:00               0           13.0   
3 2025-01-06 13:56:00 2025-01-06 13:59:00               0            0.0   
4 2025-01-06 11:00:00 20

In [132]:
# ============================================
# PHASE 2 — TWO-STAGE FLIGHT DELAY PREDICTION
# (Leakage-free + Improved Features)
# ============================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------------------------------
# Step 1: Load Dataset
# -------------------------------
df = pd.read_csv("cleaned_flights.csv")

print("Dataset shape:", df.shape)

# -------------------------------
# Step 2: Create New Feature
# -------------------------------
df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100

# -------------------------------
# Step 3: Handle Missing Values
# -------------------------------
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = ['OP_UNIQUE_CARRIER','ORIGIN','DEST','TAIL_NUM']

df[num_cols] = df[num_cols].fillna(0)
df[cat_cols] = df[cat_cols].fillna('unknown')

# -------------------------------
# Step 4: Encode Categorical
# -------------------------------
encoders = {}

for col in cat_cols:
    
    le = LabelEncoder()
    
    df[col] = le.fit_transform(df[col].astype(str))
    
    encoders[col] = le

# -------------------------------
# Step 5: Feature Selection
# (NO DATA LEAKAGE)
# -------------------------------
features = [

"YEAR",
"DAY_OF_WEEK",
"CRS_DEP_TIME",
"DEP_HOUR",
"DISTANCE",
"ORIGIN_AIRPORT_ID",
"DEST_AIRPORT_ID",
"OP_UNIQUE_CARRIER",
"ORIGIN",
"DEST",
"TAIL_NUM"

]

# -------------------------------
# Stage 1 Target
# -------------------------------
df["DELAY15"] = (df["ARR_DELAY"] >= 15).astype(int)

X1 = df[features]
y1 = df["DELAY15"]

# -------------------------------
# Train / Test Split
# -------------------------------
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1,
    y1,
    test_size=0.2,
    random_state=42
)

print("Stage 1 train size:", X1_train.shape)
print("Stage 1 test size:", X1_test.shape)

# -------------------------------
# Stage 1 Model
# -------------------------------
stage1_model = HistGradientBoostingClassifier(

    max_iter=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    class_weight={0:1,1:4}

)

stage1_model.fit(X1_train, y1_train)

print("Stage 1 model trained.")

# -------------------------------
# Stage 1 Evaluation
# -------------------------------
y1_pred = stage1_model.predict(X1_test)

print("\nStage 1 Metrics:")

print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("Precision:", precision_score(y1_test, y1_pred))
print("Recall:", recall_score(y1_test, y1_pred))
print("F1 Score:", f1_score(y1_test, y1_pred))

# -------------------------------
# Stage 2 Dataset
# (Only delayed flights)
# -------------------------------
delayed_idx = df["ARR_DELAY"] > 0

X2 = df.loc[delayed_idx, features]

# LOG transform
y2 = np.log1p(df.loc[delayed_idx, "ARR_DELAY"])

# -------------------------------
# Stage 2 Split
# -------------------------------
X2_train, X2_test, y2_train, y2_test = train_test_split(

    X2,
    y2,
    test_size=0.2,
    random_state=42

)

# -------------------------------
# Stage 2 Model
# -------------------------------
stage2_model = HistGradientBoostingRegressor(

    max_iter=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42

)

stage2_model.fit(X2_train, y2_train)

print("Stage 2 model trained.")

# -------------------------------
# Stage 2 Evaluation
# -------------------------------
y2_pred = stage2_model.predict(X2_test)

# Convert back to minutes
y2_pred = np.expm1(y2_pred)
y2_test = np.expm1(y2_test)

mae = mean_absolute_error(y2_test, y2_pred)

rmse = np.sqrt(mean_squared_error(y2_test, y2_pred))

within_10 = np.mean(np.abs(y2_test - y2_pred) <= 10) * 100

print("\nStage 2 Metrics:")

print("MAE:", mae)
print("RMSE:", rmse)
print("% within 10 minutes:", within_10)

# -------------------------------
# Save Models for Phase 3
# -------------------------------
joblib.dump(stage1_model,"model_dep.pkl")
joblib.dump(stage2_model,"model_arr.pkl")

joblib.dump(features,"feature_list.pkl")
joblib.dump(encoders,"encoders.pkl")

print("\nModels and feature encoders saved successfully for Phase 3 app!")


Dataset shape: (539714, 40)
Stage 1 train size: (431771, 11)
Stage 1 test size: (107943, 11)
Stage 1 model trained.

Stage 1 Metrics:
Accuracy: 0.6644525351342838
Precision: 0.2781218995521948
Recall: 0.5272736515682985
F1 Score: 0.3641598202373429
Stage 2 model trained.

Stage 2 Metrics:
MAE: 34.799379778203665
RMSE: 89.18385270588955
% within 10 minutes: 35.890657680687895

Models and feature encoders saved successfully for Phase 3 app!
